# 06 â€” Analysis

Score logged runs, compute per-task metrics, regenerate `site/results.html`.

**Task C's false-positive rate on clean items is the headline number.** Report it prominently even when it is unflattering â€” that is the finding the benchmark exists to produce.

Always report the paraphrased-subset score alongside the verbatim score. Never put a number on the site that does not trace to a run manifest.

Read the `running-evals` and `publishing-results` skills.

In [ ]:
# Colab bootstrap. Run once per runtime.
!pip install -q google-cloud-storage pandas

from google.colab import auth
auth.authenticate_user()

import sys, pathlib
GITHUB_USER = 'ajrngn'
REPO = pathlib.Path('/content/auditagent-bench')
if not REPO.exists():
    !git clone -q https://github.com/{GITHUB_USER}/auditagent-bench.git {REPO}
sys.path.insert(0, str(REPO))

In [ ]:
# --- Config. Every tunable value in this notebook lives in this cell. ---
BUCKET = 'audit-agent-500113-data'
RESULTS_PREFIX = 'results'
RUN_IDS = []           # TODO: run_ids to include in this results version

In [ ]:
from src import gcs
import pandas as pd

runs = []
for run_id in RUN_IDS:
    manifest = gcs.read_json(BUCKET, f'{RESULTS_PREFIX}/{run_id}/manifest.json')
    scored = list(gcs.read_jsonl(BUCKET, f'{RESULTS_PREFIX}/{run_id}/scored.jsonl'))
    runs.append((manifest, scored))

# Results from different prompt versions are not comparable.
versions = {(m['prompt_version'], m['prompt_sha256']) for m, _ in runs}
assert len(versions) <= 1, f'mixed prompt versions: {versions}'
print(f'{len(runs)} runs, prompt {versions}')

In [ ]:
def task_c_metrics(scored):
    """False-positive rate on clean items is the headline metric."""
    clean = [s for s in scored if not s['label']['is_deficient']]
    deficient = [s for s in scored if s['label']['is_deficient']]

    fp = sum(1 for s in clean if s['parsed'] and s['parsed'].get('is_deficient'))
    tp = sum(1 for s in deficient if s['parsed'] and s['parsed'].get('is_deficient'))

    return {
        'fp_rate_clean': fp / len(clean) if clean else None,
        'recall_deficient': tp / len(deficient) if deficient else None,
        'n_clean': len(clean),
        'n_deficient': len(deficient),
        'parse_failed': sum(1 for s in scored if s['parse_error']),
    }


rows = [{'model': m['model_string'], 'vendor': m['vendor'], **task_c_metrics(s)}
        for m, s in runs if m['task'] == 'C']
pd.DataFrame(rows).sort_values('fp_rate_clean')

In [ ]:
# TODO: severity confusion matrix (Task A), gap-type accuracy (Task B),
# citation precision and hallucinated-citation rate (Task D).
#
# TODO: paraphrased-subset delta â€” score paraphrased:true items separately and
# report the gap against their verbatim twins. A large gap indicates
# memorization rather than reasoning.

In [ ]:
# TODO: regenerate site/results.html from the frame above.
#
# results.html is a GENERATED file â€” never hand-edited â€” so a published number
# cannot drift from its manifest. Each row links to its run manifest.
# The table contract is documented in the committed site/results.html.